In [3]:
from pathlib import Path
import json, csv
from datetime import date, timedelta

PROJECT = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd().parents[2]
RAW = PROJECT / "data" / "raw" / "tomtom_stats"
INPUT_JSON = RAW / "7987791_HCMC_D1_Weekday_15min_unzipped.json"

PROC = PROJECT / "data" / "processed" / "tomtom_stats" / "times"
PROC.mkdir(parents=True, exist_ok=True)

print(INPUT_JSON.exists(), INPUT_JSON)


True C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\tomtom_stats\7987791_HCMC_D1_Weekday_15min_unzipped.json


In [4]:
# ----- đọc JSON
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

date_ranges = {dr["@id"]: dr for dr in data["dateRanges"]}
time_sets = {ts["@id"]: ts for ts in data["timeSets"]}
segments = data["network"]["segmentResults"]


In [5]:
# ----- sinh danh sách ngày hợp lệ theo dateRange (loại trừ weekend nếu có)
def list_days(dr):
    y1,m1,d1 = map(int, dr["from"].split("-"))
    y2,m2,d2 = map(int, dr["to"].split("-"))
    cur = date(y1,m1,d1)
    end = date(y2,m2,d2)
    ex_week = set(dr.get("excludedDaysOfWeek", []))  # ["SATURDAY","SUNDAY"]
    days = []
    while cur <= end:
        dow = cur.strftime("%A").upper()   # "MONDAY"...
        if dow not in ex_week:
            days.append((cur.isoformat(), dow))
        cur += timedelta(days=1)
    return days

days_by_dr = {drid: list_days(dr) for drid,dr in date_ranges.items()}
sum(len(v) for v in days_by_dr.values())


22

In [6]:
# ----- ánh xạ timeSet -> day-of-week mà timeSet áp dụng (MON..FRI)
# Trong JSON: timeSets[id]['dayToTimeRanges'] cho biết dayOfWeek và "06:00-06:15",...
timeset_to_valid_dow = {}
timeset_to_timerange = {}
for tid, ts in time_sets.items():
    dows = [d['dayOfWeek'].upper() for d in ts["dayToTimeRanges"]]
    # tất cả timeRanges trong timeGroups của TomTom là list, ở đây luôn 1 phần tử (slot 15')
    # lấy time-range từ name 'T0600_0615' hoặc từ dayToTimeRanges[0]['timeRanges'][0]
    tr = ts["dayToTimeRanges"][0]["timeRanges"][0]
    timeset_to_valid_dow[int(tid)] = set(dows)
    timeset_to_timerange[int(tid)] = tr

# helper để tách "HH:MM-HH:MM"
def split_tr(tr):
    start, end = tr.split("-")
    return start, end


In [7]:
# ----- flatten ra per (date, timeSet, segment)
OUT = PROC / "observations.csv"
with open(OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "date","day_of_week","time_set_id","time_start","time_end",
        "segment_id","streetName","frc","speedLimit","distance",
        "sampleSize","averageSpeed","medianSpeed","harmonicAverageSpeed",
        "averageTravelTime","medianTravelTime","travelTimeRatio",
        # percentiles rút gọn (p05,p25,p50,p75,p95) nếu có đủ 19 phần tử
        "p05","p25","p50","p75","p95",
        "is_observed"
    ])

    for seg in segments:
        sid = seg["segmentId"]
        street = seg.get("streetName","")
        frc = seg.get("frc","")
        sl = seg.get("speedLimit","")
        dist = seg.get("distance","")
        for tr in seg["segmentTimeResults"]:
            tid = tr["timeSet"]
            drid = tr["dateRange"]
            # ngày nào được phép theo dateRange & trùng dayOfWeek với timeSet
            trange = timeset_to_timerange[tid]
            ts, te = split_tr(trange)
            valid_dow = timeset_to_valid_dow[tid]
            for d, dow in days_by_dr[drid]:
                if dow in valid_dow:
                    sp = tr.get("speedPercentiles", []) or []
                    # chọn p05,p25,p50,p75,p95 nếu có
                    p05 = sp[2] if len(sp)>=3 else ""
                    p25 = sp[5] if len(sp)>=6 else ""
                    p50 = sp[9] if len(sp)>=10 else ""
                    p75 = sp[13] if len(sp)>=14 else ""
                    p95 = sp[17] if len(sp)>=18 else ""
                    sample = tr.get("sampleSize", 0)
                    w.writerow([
                        d, dow, tid, ts, te,
                        sid, street, frc, sl, dist,
                        tr.get("sampleSize",""),
                        tr.get("averageSpeed",""),
                        tr.get("medianSpeed",""),
                        tr.get("harmonicAverageSpeed",""),
                        tr.get("averageTravelTime",""),
                        tr.get("medianTravelTime",""),
                        tr.get("travelTimeRatio",""),
                        p05,p25,p50,p75,p95,
                        1 if sample and sample>0 else 0
                    ])

print("Saved:", OUT)


Saved: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\processed\tomtom_stats\times\observations.csv


In [8]:
# (tùy chọn) thêm bảng metadata time-sets & ngày để join dễ
META_TS = PROC / "time_sets.csv"
with open(META_TS, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["time_set_id","name","time_start","time_end","days"])
    for tid, ts in time_sets.items():
        tr = ts["dayToTimeRanges"][0]["timeRanges"][0]
        ts_, te_ = split_tr(tr)
        days = [d["dayOfWeek"] for d in ts["dayToTimeRanges"]]
        w.writerow([tid, ts["name"], ts_, te_, "|".join(days)])

META_DAYS = PROC / "days_by_dateRange.csv"
with open(META_DAYS, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["date_range_id","date","dow"])
    for drid, lst in days_by_dr.items():
        for d, dow in lst:
            w.writerow([drid, d, dow])

print("Saved:", META_TS, "\nSaved:", META_DAYS)


Saved: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\processed\tomtom_stats\times\time_sets.csv 
Saved: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\processed\tomtom_stats\times\days_by_dateRange.csv
